# Pengolahan Data Sensor PIR - Feature Engineering

## 1. Install dan Import Library

In [ ]:
!pip install openpyxl -q

import pandas as pd
import numpy as np
from google.colab import files
import io

print('Library berhasil diimport!')

Library berhasil diimport!


## 2. Upload File Excel

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'File diupload: {filename}')

Saving Log PIR_Dataset.xlsx to Log PIR_Dataset.xlsx
File diupload: Log PIR_Dataset.xlsx


## 3. Konfigurasi Nama Kolom

Sesuaikan nama kolom di bawah jika berbeda dengan file kamu.

In [ ]:
CONFIG = {
    'col_count'            : 'Jumlah Deteksi',
    'col_durasi'           : 'Durasi Aktif (detik)',
    'col_interval_mulai'   : 'Interval Mulai',
    'col_interval_selesai' : 'Interval Selesai',
    'col_tanggal'          : 'Tanggal',
    'header_row'           : 0,
}
print('Konfigurasi OK:', CONFIG)

Konfigurasi OK: {'col_count': 'Jumlah Deteksi', 'col_durasi': 'Durasi Aktif (detik)', 'col_interval_mulai': 'Interval Mulai', 'col_interval_selesai': 'Interval Selesai', 'col_tanggal': 'Tanggal', 'header_row': 0}


## 4. Baca dan Validasi Data

In [ ]:
df = pd.read_excel(io.BytesIO(uploaded[filename]), header=CONFIG['header_row'])
df.dropna(how='all', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Data berhasil dibaca: {len(df)} baris, {len(df.columns)} kolom')
print('\nPreview data:')
display(df.head(10))

Data berhasil dibaca: 1487 baris, 10 kolom

Preview data:


,No,Tanggal,Interval Mulai,Interval Selesai,Perangkat,Lokasi,Jumlah Deteksi,Durasi Aktif (detik),WiFi RSSI (dBm),Kualitas Sinyal
0,1,2026-05-14,2026-05-14 12:45:37,2026-05-14 12:55:37,ESP32-PIR-01,Prototipe sawah,11,94.701,-18,Sangat Baik
1,2,2026-05-14,2026-05-14 12:55:48,2026-05-14 13:05:37,ESP32-PIR-01,Prototipe sawah,6,44.100,-20,Sangat Baik
2,3,2026-05-14,2026-05-14 13:05:43,2026-05-14 13:15:37,ESP32-PIR-01,Prototipe sawah,4,35.000,-21,Sangat Baik
3,4,2026-05-14,2026-05-14 13:15:44,2026-05-14 13:25:37,ESP32-PIR-01,Prototipe sawah,2,15.500,-20,Sangat Baik
4,5,2026-05-14,2026-05-14 13:25:45,2026-05-14 13:35:37,ESP32-PIR-01,Prototipe sawah,3,26.300,-18,Sangat Baik
5,6,2026-05-14,2026-05-14 13:35:45,2026-05-14 13:45:37,ESP32-PIR-01,Prototipe sawah,4,34.801,-12,Sangat Baik
6,7,2026-05-14,2026-05-14 13:45:49,2026-05-14 13:55:37,ESP32-PIR-01,Prototipe sawah,5,43.902,-14,Sangat Baik
7,8,2026-05-14,2026-05-14 13:55:44,2026-05-14 14:05:37,ESP32-PIR-01,Prototipe sawah,17,143.807,-11,Sangat Baik
8,9,2026-05-14,2026-05-14 14:05:46,2026-05-14 14:15:37,ESP32-PIR-01,Prototipe sawah,17,149.302,-13,Sangat Baik
9,10,2026-05-14,2026-05-14 14:15:50,2026-05-14 14:25:37,ESP32-PIR-01,Prototipe sawah,1,8.700,-14,Sangat Baik


## 5. Preprocessing

In [ ]:
col_count  = CONFIG['col_count']
col_durasi = CONFIG['col_durasi']
col_mulai  = CONFIG['col_interval_mulai']
col_selesai= CONFIG['col_interval_selesai']
col_tanggal= CONFIG['col_tanggal']

df[col_count]  = pd.to_numeric(df[col_count],  errors='coerce').fillna(0).astype(int)
df[col_durasi] = pd.to_numeric(df[col_durasi], errors='coerce').fillna(0)

for col_time in [col_mulai, col_selesai, col_tanggal]:
    if col_time in df.columns:
        df[col_time] = pd.to_datetime(df[col_time], errors='coerce')

if col_mulai in df.columns:
    df.sort_values(col_mulai, inplace=True)
    df.reset_index(drop=True, inplace=True)

print('Preprocessing selesai')
display(df[col_count].describe())

Preprocessing selesai


,Jumlah Deteksi
count,1487.000000
mean,5.281103
std,8.159270
min,0.000000
25%,0.000000
50%,1.000000
75%,8.000000
max,40.000000


## 6. Feature Engineering: t (interval), Lag, Avg, Delta

In [ ]:
# t (interval): nomor urut interval mulai dari 1
df['t (interval)'] = range(1, len(df) + 1)

# Lag 1 sampai 6
for lag in range(1, 7):
    df[f'lag{lag}'] = df[col_count].shift(lag).fillna(0).astype(int)

# avg30 = rata-rata 3 interval terakhir
df['avg30'] = df[col_count].rolling(window=3, min_periods=1).mean().round(3)

# avg60 = rata-rata 6 interval terakhir
df['avg60'] = df[col_count].rolling(window=6, min_periods=1).mean().round(3)

# delta1 = selisih count saat ini vs sebelumnya
df['delta1'] = df[col_count].diff().fillna(0).astype(int)

print('Fitur berhasil dihitung!')
display(df[['t (interval)', col_count, 'lag1', 'lag2', 'lag3', 'avg30', 'avg60', 'delta1']].head(10))

Fitur berhasil dihitung!


,t (interval),Jumlah Deteksi,lag1,lag2,lag3,avg30,avg60,delta1
0,1,11,0,0,0,11.000,11.000,0
1,2,6,11,0,0,8.500,8.500,-5
2,3,4,6,11,0,7.000,7.000,-2
3,4,2,4,6,11,4.000,5.750,-2
4,5,3,2,4,6,3.000,5.200,1
5,6,4,3,2,4,3.000,5.000,1
6,7,5,4,3,2,4.000,4.000,1
7,8,17,5,4,3,8.667,5.833,12
8,9,17,17,5,4,13.000,8.000,0
9,10,1,17,17,5,11.667,7.833,-16


## 7. Labeling Aktivitas Tikus (Berdasarkan Persentil)

| Label | Kondisi | Porsi Data |
|-------|---------|------------|
| **rendah** | count <= P33 | ~33% data terkecil |
| **sedang** | P33 < count <= P67 | ~33% data tengah |
| **tinggi** | count > P67 | ~34% data tertinggi |

In [ ]:
p33 = df[col_count].quantile(0.33)
p67 = df[col_count].quantile(0.67)

print(f'Threshold Persentil:')
print(f'  P33 (batas rendah/sedang) : {p33}')
print(f'  P67 (batas sedang/tinggi) : {p67}')

def label_aktivitas(count):
    if count <= p33:
        return 'rendah'
    elif count <= p67:
        return 'sedang'
    else:
        return 'tinggi'

df['label'] = df[col_count].apply(label_aktivitas)

print('\nDistribusi Label Aktivitas:')
total = len(df)
for lbl, cnt in df['label'].value_counts().items():
    print(f'  {lbl:8s}: {cnt:4d} baris ({cnt/total*100:.1f}%)')

Threshold Persentil:
  P33 (batas rendah/sedang) : 0.0
  P67 (batas sedang/tinggi) : 4.0

Distribusi Label Aktivitas:
  rendah  :  701 baris (47.1%)
  tinggi  :  483 baris (32.5%)
  sedang  :  303 baris (20.4%)


## 8. Susun Kolom Output Sesuai Format

In [ ]:
# Susun kolom sesuai urutan: tanggal, interval mulai, interval selesai,
# t (interval), count, durasi aktif, lag1-6, avg30, avg60, delta1, label
output_cols = [
    col_tanggal,
    col_mulai,
    col_selesai,
    't (interval)',
    col_count,
    col_durasi,
    'lag1', 'lag2', 'lag3', 'lag4', 'lag5', 'lag6',
    'avg30', 'avg60',
    'delta1',
    'label',
]

# Rename kolom count dan durasi ke nama pendek
df_out = df[output_cols].copy()
df_out = df_out.rename(columns={
    col_count  : 'count',
    col_durasi : 'durasi aktif',
    col_tanggal: 'Tanggal',
    col_mulai  : 'Interval Mulai',
    col_selesai: 'Interval Selesai',
})

print('Preview hasil akhir (15 baris pertama):')
display(df_out.head(15))
print(f'\nTotal baris : {len(df_out)}')
print(f'Total kolom : {len(df_out.columns)}')

Preview hasil akhir (15 baris pertama):


,Tanggal,Interval Mulai,Interval Selesai,t (interval),count,durasi aktif,lag1,lag2,lag3,lag4,lag5,lag6,avg30,avg60,delta1,label
0,2026-05-14,2026-05-14 12:45:37,2026-05-14 12:55:37,1,11,94.701,0,0,0,0,0,0,11.000,11.000,0,tinggi
1,2026-05-14,2026-05-14 12:55:48,2026-05-14 13:05:37,2,6,44.100,11,0,0,0,0,0,8.500,8.500,-5,tinggi
2,2026-05-14,2026-05-14 13:05:43,2026-05-14 13:15:37,3,4,35.000,6,11,0,0,0,0,7.000,7.000,-2,sedang
3,2026-05-14,2026-05-14 13:15:44,2026-05-14 13:25:37,4,2,15.500,4,6,11,0,0,0,4.000,5.750,-2,sedang
4,2026-05-14,2026-05-14 13:25:45,2026-05-14 13:35:37,5,3,26.300,2,4,6,11,0,0,3.000,5.200,1,sedang
5,2026-05-14,2026-05-14 13:35:45,2026-05-14 13:45:37,6,4,34.801,3,2,4,6,11,0,3.000,5.000,1,sedang
6,2026-05-14,2026-05-14 13:45:49,2026-05-14 13:55:37,7,5,43.902,4,3,2,4,6,11,4.000,4.000,1,tinggi
7,2026-05-14,2026-05-14 13:55:44,2026-05-14 14:05:37,8,17,143.807,5,4,3,2,4,6,8.667,5.833,12,tinggi
8,2026-05-14,2026-05-14 14:05:46,2026-05-14 14:15:37,9,17,149.302,17,5,4,3,2,4,13.000,8.000,0,tinggi
9,2026-05-14,2026-05-14 14:15:50,2026-05-14 14:25:37,10,1,8.700,17,17,5,4,3,2,11.667,7.833,-16,sedang



Total baris : 1487
Total kolom : 16


## 9. Simpan dan Download File Output

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment

# Correctly generate the output filename by checking the extension once.
if filename.endswith('.xlsx'):
    output_filename = filename.replace('.xlsx', '_HASIL_OLAH.xlsx')
elif filename.endswith('.xls'):
    output_filename = filename.replace('.xls', '_HASIL_OLAH.xlsx')
else:
    # Fallback for unexpected extensions, append .xlsx
    output_filename = filename + '_HASIL_OLAH.xlsx'

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:

    df_out.to_excel(writer, sheet_name='Data_Fitur', index=False)

    summary = {
        'Keterangan': [
            'Total Data (baris)', 'Nilai Count Minimum', 'Nilai Count Maximum',
            'Rata-rata Count', 'Median Count', '',
            '--- THRESHOLD PERSENTIL ---',
            'P33 (batas rendah / sedang)', 'P67 (batas sedang / tinggi)', '',
            '--- DISTRIBUSI LABEL ---',
            'Jumlah Label rendah', 'Jumlah Label sedang', 'Jumlah Label tinggi',
            'Persentase rendah (%)', 'Persentase sedang (%)', 'Persentase tinggi (%)',
        ],
        'Nilai': [
            len(df_out), df_out['count'].min(), df_out['count'].max(),
            round(df_out['count'].mean(), 3), df_out['count'].median(), '',
            '', round(p33, 3), round(p67, 3), '',
            '',
            (df_out['label'] == 'rendah').sum(),
            (df_out['label'] == 'sedang').sum(),
            (df_out['label'] == 'tinggi').sum(),
            round((df_out['label'] == 'rendah').mean() * 100, 1),
            round((df_out['label'] == 'sedang').mean() * 100, 1),
            round((df_out['label'] == 'tinggi').mean() * 100, 1),
        ],
    }
    pd.DataFrame(summary).to_excel(writer, sheet_name='Ringkasan', index=False)

    wb = writer.book
    feature_set = {'lag1','lag2','lag3','lag4','lag5','lag6','avg30','avg60','delta1','label','t (interval)'}
    for sheet_name in ['Data_Fitur', 'Ringkasan']:
        ws = wb[sheet_name]
        for cell in ws[1]:
            fill_color = '2E75B6' if cell.value in feature_set else '1F4E79'
            cell.fill = PatternFill('solid', start_color=fill_color)
            cell.font = Font(bold=True, color='FFFFFF', name='Arial', size=10)
            cell.alignment = Alignment(horizontal='center', wrap_text=True)
        for col in ws.columns:
            max_len = max((len(str(c.value)) for c in col if c.value is not None), default=10)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 32)

print(f'File disimpan: {output_filename}')
files.download(output_filename)
print('File sedang didownload ke komputer kamu...')


File disimpan: Log PIR_Dataset_HASIL_OLAH.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File sedang didownload ke komputer kamu...
